In [1]:
%run cochain_complex.ipynb
import copy
import numbers
import decimal
from sympy import *
from multiset import *
from itertools import combinations
import time 

In [2]:
T=T_symb(7)

## Normal Forms and Derivatives

In [3]:
def normal_form(ind_expr):
    '''arg: ind_expr, a polynomial of indexed objects which each 
            represent structure functions for the symplectified distribution
            (ex: P[2,1,3,3,8,9] is the 88 derivative of 2-cochain P[1,3,3])
       returns: The normal form of the expression, exchanging derivatives as needed'''
    if ind_expr in NF_dict:
        return NF_dict[ind_expr]
    if type(ind_expr)==Add:
        result = simplify(Add(*[normal_form(A) for A in ind_expr.args]))
        NF_dict[ind_expr] = result
        return result
    if type(ind_expr)==Mul:
        result=simplify(Mul(*[normal_form(A) for A in ind_expr.args]))
        NF_dict[ind_expr] = result
        return result
    if isinstance(ind_expr,numbers.Number):
        NF_dict[ind_expr]=ind_expr
        return ind_expr
    if type(ind_expr)==Symbol:
        return ind_expr
    if type(ind_expr)==Pow:
        return normal_form(ind_expr.base)**normal_form(ind_expr.exp)
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        deg=ind_expr.indices[0]
        im_ind=ind_expr.indices[1:2]
        ind=ind_expr.indices[2:deg+2]
        ders=ind_expr.indices[deg+2:len(ind_expr.indices)]
        
        if list(ind)!=sorted(ind):
            new_ind=sorted(ind)
            sgn=perm_sign(ind,new_ind)
            L=[deg]+list(im_ind)+list(new_ind)+list(ders)
            result=simplify(normal_form(sgn*base[L]))
            NF_dict[ind_expr]=result
            return result
        
        if list(ders)==sorted(ders):
            NF_dict[ind_expr]=ind_expr
            return ind_expr
        j=0 # Least int so that der[j]>der[j+1]
        while ders[j]<=ders[j+1]:
            j++1
        result=base[[deg]+list(im_ind)+list(ind)+list(ders[0:j])+[ders[j+1],ders[j]]]
        for l in range(2*m+5):
            result=result+base[2,l,ders[j+1],ders[j]]*base[[deg]+list(im_ind)+list(ind)+list(ders[0:j])+[l]]
        for i in ders[j+2:len(ders)]:
            result=ind_der(result,i)
        result=simplify(result)
        NF_dict[ind_expr]=result
        return result
        

In [4]:
def ind_der(ind_expr,i):
    '''args: ind_expr, an expression in coordinates h,e,y, and indexed objects, and a natural number i
       returns: the normal form of the derivative of the expression in the i direction'''
    # I'm not sure if this simplification will help or hurt time efficiency
    ind_expr=simplify(ind_expr)
    if isinstance(ind_expr,numbers.Number):
        return 0
    if type(ind_expr)==Add:
        result = Add(*[abn_ind_der(A,i) for A in ind_expr.args])
        return normal_form(result)
    if type(ind_expr)==Mul:
        result=0
        for j in range(len(ind_expr.args)):
            result+=abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
        return normal_form(result)
    # Here I assume the exponents are constant
    if type(ind_expr)==Pow:
        return normal_form(ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*abn_ind_der(ind_expr.base,i))
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        ind=list(ind_expr.indices)
        return normal_form(base[ind+[i]])
    if type(ind_expr)==Symbol:
        if ind_expr in [y,h,e] and [y,h,e].index(ind_expr)==i: return 1
        else: return 0
    
def abn_ind_der(ind_expr,i):
    '''args: ind_expr, an expression in coordinates h,e,y, and indexed objects, and a natural number i
       returns: the derivative of the expression in the i direction without reducing to normal form'''
    # I'm not sure if this simplification will help or hurt time efficiency
    ind_expr=simplify(ind_expr)
    if isinstance(ind_expr,numbers.Number):
        return 0
    if type(ind_expr)==Add:
        result = Add(*[abn_ind_der(A,i) for A in ind_expr.args])
        return result
    if type(ind_expr)==Mul:
        result=0
        for j in range(len(ind_expr.args)):
            result+=abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
        return result
    if type(ind_expr)==Pow:
        return ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*abn_ind_der(ind_expr.base,i)
    if type(ind_expr)==Indexed:
        base=ind_expr.base
        ind=list(ind_expr.indices)
        return base[ind+[i]]
    if type(ind_expr)==Symbol:
        if ind_expr in [y,h,e] and [y,h,e].index(ind_expr)==i: return 1
        else: return 0
    if type(ind_expr)==exp:
        return ind_expr*abn_ind_der(ind_expr.args[0],i)

In [5]:
def str_func(T,fm,sf):
    '''arg: T, a T_symb object
            fm, a matrix whose columns represent an algebraic frame in the basis of T_symb;
                that is, it is block upper triangular (with non)
            sf, the structure function for the basis
       returns: the structure function for this frame represented by fm, as a cochain'''
    r_dict={}
    for i in range(len(T.basis)):
        A_vec=fm.col(i)
        A=T.elt(A_vec)
        for j in range(i,len(T.basis)):
            B_vec=fm.col(j)
            B=T.elt(B_vec)
            w=Matrix(SF_ad(A,B,sf).vec_rep)
            for k in reversed(list(range(len(T.basis)))):
                if fm[k,k]!=0:
                    coeff=w[k]/fm[k,k]
                    r_dict[(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])]=coeff
                    w+=Matrix([-coeff*A for A in fm.col(k)])
    return T.cochain_complex.cochain(r_dict)

In [6]:
def SF_ad(X1,X2,SF):
    '''args: X1,X2 are VFs, represented as linear combinations from the T basis with
               coeffs among the base SFs P and the coordinates y,h,e
             SF is the structure function defining the ad-relations between T basis elts,
               an element of C, the cochain complex
       returns: [X1,X2], where the str function defines the relations between T basis elts,
               and P,y,h,e depend on the coordinates appropriately'''
    
    result=0
    for i in range(len(X1.vec_rep)):
        coeff_1=X1.vec_rep[i]
        if coeff_1!=0:
            for j in range(len(X2.vec_rep)):
                coeff_2=X2.vec_rep[j]
                if coeff_2!=0:
                    res1=coeff_1*abn_ind_der(coeff_2,i)*T.basis[j]
                    res2=-coeff_2*abn_ind_der(coeff_1,j)*T.basis[i]
                    # For testing; can remove later
#                     print('\n',(i,j))
#                     print('X1 cmpnt:', coeff_1*T.basis[i])
#                     print('X2 cmpnt:', coeff_2*T.basis[j])
#                     print('res1:',res1,'of type',type(res1))
#                     print('res2:',res2,'of type',type(res2))
                    new_res=result+res1+res2
                    result+=res1
                    result+=res2
#                     print('curr_result:', result)
    e_elt=e_elt=X1.cast_as_ext_elt().wedge(X2.cast_as_ext_elt())
#     print('\nFinal res:',SF.apply_cochain_map(e_elt))
    result+=SF.apply_cochain_map(e_elt)
    return result

In [25]:
def simplify_cochain(c):
    time0=time.time()
    ctr=0
    print('Total coeffs:',len(list(c.coeff_dict.keys())))
    for k in c.coeff_dict:
        print(ctr,'coeffs simplified; prev. time', round(time.time()-time0,2))
        time0=time.time()
        c.coeff_dict[k]=simplify(c.coeff_dict[k])
        ctr+=1

## Computing Prolongations

We'll start with a distribution with structure function K, then prolong and normalize.
The bundle P0 will have vertical coordinates h,e;
The bundle P1 will have vertical coordinate y

In [19]:
NF_dict={}
m=3
n=m+3
K=IndexedBase('K')
h,e,y=symbols('h,e,y')
T=T_symb(2*m+1)
C=T.cochain_complex

In [20]:
# Start with the structure function on the base manifold; the H,E,Y components don't have meaning here
K0=C.cochain({})
C.init_basis(2)
for wght in C.basis[2]:
    if wght>=0:
        for c in C.basis[2][wght]:
            t=C.ijk_triple(c)
            if t[0]>2 and t[1]>2 and t[2]>2:
                A=[T.basis_strs[ti] for ti in t]
                K0=K0+C.cochain({(A[0],A[1],A[2]):K[t[0],t[1],t[2]]})
                
# bracket relations (degree zero part of structure function)
br_rels={}
br_rels.update({K[3,a,a+1]:1 for a in range(4,2*n-3)}) # [X,ei]
br_rels.update({K[a,2*n+1-a,2*n-2]:(-1)**(1+a) for a in range(4,n+1)}) # [ei, e_(2n+5-i)]
br_rels.update({K[a,b,a+b]:0 for b in range(4,2*n-2) for a in range(2*n-2-b)}) #[ei,ej], i+j<2n-5

K0=K0.subs(br_rels)

In [21]:
F0=eye(len(T.basis))
F0[0,0]=0
#Lift invariantly
F1=F0*T.Ad_mat(h*T.basis[1]+e*T.basis[2])
K1=str_func(T,F1,K0)
K1_p1=C.subspace_proj(K1,'im')[1]

In [ ]:
# Sanity check: is K1 the str_func of F1?

for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        ad_ij1=SF_ad(T.elt(list(F1.col(i))),T.elt(list(F1.col(j))),K0)
        w=zeros(len(T.basis),1)
        for k in range(len(T.basis)):
            key=(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])
            if key in K1.coeff_dict: 
                w+=F1.col(k)*K1.coeff_dict[key]
        ad_ij2=sum([w[l]*T.basis[l] for l in range(len(T.basis))])
        result=T.elt([simplify(ad_ij1.vec_rep[l]-ad_ij2.vec_rep[l]) for l in range(len(T.basis))])
        if result!=T.elt([0]*len(T.basis)):
            print('\n\n\nFailure at', (i,j),':')
            print('ad_ij1 =', ad_ij1,'\n\nad_ij2 =',ad_ij2)
            print('\nDifference:',result)

In [22]:
# Normalize
f1=C.preim_elt(K1_p1)
f1_Mat=f1.find_mat_rep()

In [23]:
F1_N=F1*(eye(len(T.basis))+f1_Mat)
K1_N=str_func(T,F1_N,K0)

In [27]:
simplify_cochain(K1)
simplify_cochain(K1_N)
df1=f1.coboundary()
simplify_cochain(df1)

Total coeffs: 210
0 coeffs simplified; prev. time 0.0
1 coeffs simplified; prev. time 0.0
2 coeffs simplified; prev. time 0.0
3 coeffs simplified; prev. time 0.0
4 coeffs simplified; prev. time 0.0
5 coeffs simplified; prev. time 0.0
6 coeffs simplified; prev. time 0.0
7 coeffs simplified; prev. time 0.0
8 coeffs simplified; prev. time 0.0
9 coeffs simplified; prev. time 0.0
10 coeffs simplified; prev. time 0.0
11 coeffs simplified; prev. time 0.0
12 coeffs simplified; prev. time 0.0
13 coeffs simplified; prev. time 0.0
14 coeffs simplified; prev. time 0.0
15 coeffs simplified; prev. time 0.01
16 coeffs simplified; prev. time 0.01
17 coeffs simplified; prev. time 0.02
18 coeffs simplified; prev. time 0.0
19 coeffs simplified; prev. time 0.01
20 coeffs simplified; prev. time 0.03
21 coeffs simplified; prev. time 0.02
22 coeffs simplified; prev. time 0.0
23 coeffs simplified; prev. time 0.01
24 coeffs simplified; prev. time 0.01
25 coeffs simplified; prev. time 0.03
26 coeffs simplified;

3 coeffs simplified; prev. time 0.83
4 coeffs simplified; prev. time 0.19
5 coeffs simplified; prev. time 0.0
6 coeffs simplified; prev. time 0.06
7 coeffs simplified; prev. time 0.43
8 coeffs simplified; prev. time 0.13
9 coeffs simplified; prev. time 0.17
10 coeffs simplified; prev. time 0.99
11 coeffs simplified; prev. time 10.15
12 coeffs simplified; prev. time 2.82
13 coeffs simplified; prev. time 0.1
14 coeffs simplified; prev. time 0.38
15 coeffs simplified; prev. time 0.98
16 coeffs simplified; prev. time 0.93
17 coeffs simplified; prev. time 1.99
18 coeffs simplified; prev. time 1.33
19 coeffs simplified; prev. time 0.1
20 coeffs simplified; prev. time 0.35
21 coeffs simplified; prev. time 1.59
22 coeffs simplified; prev. time 3.91
23 coeffs simplified; prev. time 5.96
24 coeffs simplified; prev. time 1892.85
25 coeffs simplified; prev. time 5.24
26 coeffs simplified; prev. time 0.11
27 coeffs simplified; prev. time 0.49
28 coeffs simplified; prev. time 2.22
29 coeffs simplifi

KeyboardInterrupt: 

In [79]:
# Sanity check: K1_N=K1+df1
Test0=K1_N-K1-f1.coboundary()

In [95]:
list(T0.coeff_dict.keys())[2]

('e2', 'e6', 'E')

In [96]:
print(T0.coeff_dict[('e2','e6','E')])

((-105*K[3, 10, 5, 1] - 25*K[3, 9, 3, 1] - 60*K[4, 9, 4, 1] - 4*K[5, 8, 4, 1] - 64*K[5, 9, 5, 1] - 21*K[6, 7, 4, 1] - 25*K[6, 8, 5, 1] - 89*K[6, 9, 6, 1] - 25*K[7, 8, 6, 1] - 114*K[7, 9, 7, 1] - 114*K[8, 9, 8, 1] + 182*K[9, 10, 10, 1])*exp(e + 5*h)/1820 + 5*(-3*K[3, 10, 5]/52 - 5*K[3, 9, 3]/364 - 3*K[4, 9, 4]/91 - K[5, 8, 4]/455 - 16*K[5, 9, 5]/455 - 3*K[6, 7, 4]/260 - 5*K[6, 8, 5]/364 - 89*K[6, 9, 6]/1820 - 5*K[7, 8, 6]/364 - 57*K[7, 9, 7]/910 - 57*K[8, 9, 8]/910 + K[9, 10, 10]/10)*exp(e + 5*h))*(15*exp(2*e)*exp(2*h)*exp(-e - 5*h)*K[3, 10, 9]/364 + 134*exp(-e - 5*h)*exp(e - 3*h)*exp(e + 5*h)*K[5, 9, 9]/1911 - 25*exp(-e - 5*h)*exp(e - h)*exp(e + 3*h)*K[6, 8, 9]/2548 - exp(-e - 3*h)*exp(e - 5*h)*exp(e + 5*h)*K[4, 9, 8]/637 + 41*exp(-e - 3*h)*exp(e - 3*h)*exp(e + 3*h)*K[5, 8, 8]/1274 - 19*exp(-e - 3*h)*exp(e - h)*exp(e + h)*K[6, 7, 8]/2548 - 25*exp(-e - h)*exp(e - 5*h)*exp(e + 3*h)*K[4, 8, 7]/2548 + 41*exp(-e - h)*exp(e - 3*h)*exp(e + h)*K[5, 7, 7]/3822 - 25*exp(-e + h)*exp(e - 5*h)*exp(

In [89]:
T0=K1_N-K1-f1
for i in range(10):
    print('\n',T0.coeff_dict[list(T0.coeff_dict.keys())[i]])


 exp(-e + h)*exp(e - h) + exp(-e + h)*exp(e - h)/(-23*exp(2*h)*exp(-e - 5*h)*exp(e + 3*h)/176 - 5*exp(2*h)*exp(-e - 3*h)*exp(e + h)/44 - 39*exp(2*h)*exp(-e + h)*exp(e - 3*h)/44 - 153*exp(2*h)*exp(-e + 3*h)*exp(e - 5*h)/176 + 1 - 23*exp(-2*e)*exp(e - 5*h)*exp(e + 5*h)/176 + 3*exp(-2*e)*exp(e - 3*h)*exp(e + 3*h)/176 + 5*exp(-2*e)*exp(e - h)*exp(e + h)/44)

 (-((-27*K[3, 4, 4, 2] - 29*K[3, 5, 5, 2] + 12*K[3, 6, 6, 2] + 12*K[3, 7, 7, 2] + 15*K[3, 8, 8, 2] + 17*K[3, 9, 9, 2] - 2*K[5, 9, 10, 2] + 3*K[6, 8, 10, 2])*exp(e - h)/44 + (-27*K[3, 4, 4]/44 - 29*K[3, 5, 5]/44 + 3*K[3, 6, 6]/11 + 3*K[3, 7, 7]/11 + 15*K[3, 8, 8]/44 + 17*K[3, 9, 9]/44 - K[5, 9, 10]/22 + 3*K[6, 8, 10]/44)*exp(e - h) + (17*exp(2*h)*exp(-e - 5*h)*exp(e + 5*h)*K[3, 9, 9]/44 + 15*exp(2*h)*exp(-e - 3*h)*exp(e + 3*h)*K[3, 8, 8]/44 + 3*exp(2*h)*exp(-e - h)*exp(e + h)*K[3, 7, 7]/11 + 3*exp(2*h)*exp(-e + h)*exp(e - h)*K[3, 6, 6]/11 - 29*exp(2*h)*exp(-e + 3*h)*exp(e - 3*h)*K[3, 5, 5]/44 - 27*exp(2*h)*exp(-e + 5*h)*exp(e - 5*h)*K[

KeyboardInterrupt: 

In [84]:
simplify_cochain(Test0)

Total coeffs: 363
0 coeffs simplified
1 coeffs simplified
2 coeffs simplified


KeyboardInterrupt: 

In [55]:
# Sanity check: is K1_N really the str_func of F1_N?

for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        ad_ij1=SF_ad(T.elt(list(F1_N.col(i))),T.elt(list(F1_N.col(j))),K0)
        w=zeros(len(T.basis),1)
        for k in range(len(T.basis)):
            key=(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])
            if key in K1_N.coeff_dict: 
                w+=F1_N.col(k)*K1_N.coeff_dict[key]
        ad_ij2=sum([w[l]*T.basis[l] for l in range(len(T.basis))])
        result=T.elt([simplify(ad_ij1.vec_rep[l]-ad_ij2.vec_rep[l]) for l in range(len(T.basis))])
        if result!=T.elt([0]*len(T.basis)):
            print('\n\n\nFailure at', (i,j),':')
            print('ad_ij1 =', ad_ij1,'\n\nad_ij2 =',ad_ij2)
            print('\nDifference:',result)

In [62]:
simplify_cochain(K1_N_Np)
K1_N_Np

(-1260*K[3, 4, 3]*K[3, 9, 10] + 46620*K[3, 4, 4] - 25900*K[3, 5, 5] - 20720*K[3, 6, 6] - 20720*K[3, 7, 7] - 25900*K[3, 8, 8] + 4662*K[3, 9, 10]*K[4, 10, 10] - 3771*K[3, 9, 10]*K[4, 5, 5] - 2511*K[3, 9, 10]*K[4, 6, 6] - 486*K[3, 9, 10]*K[4, 7, 7] + 1539*K[3, 9, 10]*K[4, 8, 8] - 4095*K[3, 9, 10]*K[4, 9, 9] - 1377*K[3, 9, 10]*K[5, 6, 7] - 1224*K[3, 9, 10]*K[5, 7, 8] + 6894*K[3, 9, 10]*K[5, 8, 9] - 7659*K[3, 9, 10]*K[6, 7, 9] + 46620*K[3, 9, 9] + 18648*K[5, 9, 10] - 5180*K[6, 8, 10])*exp(2*h)/45584*(e2,e6,N) + (591360*K[3, 4, 3] + 181440*K[3, 4, 4]*K[4, 5, 6] + 189475*K[3, 4, 4]*K[4, 6, 7] + 154060*K[3, 4, 4]*K[4, 7, 8] + 200220*K[3, 4, 4]*K[4, 8, 9] + 5060*K[3, 4, 4]*K[5, 6, 8] - 46100*K[3, 4, 4]*K[5, 7, 9] + 194880*K[3, 5, 5]*K[4, 5, 6] + 111265*K[3, 5, 5]*K[4, 6, 7] + 73300*K[3, 5, 5]*K[4, 7, 8] - 29100*K[3, 5, 5]*K[4, 8, 9] + 9020*K[3, 5, 5]*K[5, 6, 8] - 58732*K[3, 5, 5]*K[5, 7, 9] - 80640*K[3, 6, 6]*K[4, 5, 6] + 113740*K[3, 6, 6]*K[4, 6, 7] + 58640*K[3, 6, 6]*K[4, 7, 8] - 23280*K[3, 6

In [56]:
# Sanity check: is F1_N normal up to degree 1?
K1_N_p1=K1_N.wght_proj(1)
K1_N_Np=T.cochain_complex.subspace_proj(K1_N_p1,'im')[0]
simplify_cochain(K1_N_Np)
K1_N_Np # Should be zero

In [57]:
K1_N_Np

(-1260*K[3, 4, 3]*K[3, 9, 10] + 46620*K[3, 4, 4] - 25900*K[3, 5, 5] - 20720*K[3, 6, 6] - 20720*K[3, 7, 7] - 25900*K[3, 8, 8] + 4662*K[3, 9, 10]*K[4, 10, 10] - 3771*K[3, 9, 10]*K[4, 5, 5] - 2511*K[3, 9, 10]*K[4, 6, 6] - 486*K[3, 9, 10]*K[4, 7, 7] + 1539*K[3, 9, 10]*K[4, 8, 8] - 4095*K[3, 9, 10]*K[4, 9, 9] - 1377*K[3, 9, 10]*K[5, 6, 7] - 1224*K[3, 9, 10]*K[5, 7, 8] + 6894*K[3, 9, 10]*K[5, 8, 9] - 7659*K[3, 9, 10]*K[6, 7, 9] + 46620*K[3, 9, 9] + 18648*K[5, 9, 10] - 5180*K[6, 8, 10])*exp(2*h)/45584*(e2,e6,N) + (591360*K[3, 4, 3] + 181440*K[3, 4, 4]*K[4, 5, 6] + 189475*K[3, 4, 4]*K[4, 6, 7] + 154060*K[3, 4, 4]*K[4, 7, 8] + 200220*K[3, 4, 4]*K[4, 8, 9] + 5060*K[3, 4, 4]*K[5, 6, 8] - 46100*K[3, 4, 4]*K[5, 7, 9] + 194880*K[3, 5, 5]*K[4, 5, 6] + 111265*K[3, 5, 5]*K[4, 6, 7] + 73300*K[3, 5, 5]*K[4, 7, 8] - 29100*K[3, 5, 5]*K[4, 8, 9] + 9020*K[3, 5, 5]*K[5, 6, 8] - 58732*K[3, 5, 5]*K[5, 7, 9] - 80640*K[3, 6, 6]*K[4, 5, 6] + 113740*K[3, 6, 6]*K[4, 6, 7] + 58640*K[3, 6, 6]*K[4, 7, 8] - 23280*K[3, 6

## Prenormalization

From the involutivity conditions $[V_i,V_i]\subseteq V_i$ and $[\mathcal{J}^{(i)},V_i]\subseteq \mathcal{J}^{(i)}$, we can impose the following conditions on the structure function $c_{ij}^k$:

$$ [\varepsilon_i,\varepsilon_j]=\sum_{k=1}^j c_{ij}^k\varepsilon_k,\quad \text{for}\ i<j\leq n-3$$
$$ [\varepsilon_i,\varepsilon_j]=c_{ij}^XX+\sum_{k=1}^j c_{ij}^k\varepsilon_k,\quad \text{for}\ j\geq n-3\ \text{and}\ i+j<2n-5$$
Also, from the bracket relations in the modulus, we obtain $c_{Xi}^{i+1}=1$ and $c_{i(2n+5-i)}^\eta = (-1)^i$. All this is captured in the below "prenormalization", but the indices are shifted to reflect the index of each element in the basis $(Y,H,E,X,\varepsilon_1,\ldots \varepsilon_6,\eta)$.

In [ ]:
# We can impose the following conditions on the structure function P
prenorm_dict={}

# involutivity relations
prenorm_dict.update({P[a,b,c]:0 for c in range(4,2*n-2)
                          for b in range(4,c) for a in range(4,b)})
prenorm_dict.update({P[a,b,3]:0 for b in range(4,2*n-2) for a in range(4,b)})

In [ ]:
CH_1=CH_1.subs(prenorm_dict)
g,g_coords=C7.subspace_proj(CH_1,'im')
f=C7.preim_elt(g_coords)
CH_2 = CH_1-g #CH_2 is normal

In [ ]:
# Let's make a sanity check:
print(C7.coboundary(f)==g)
print(C7.subspace_proj(g,'im')[0]==g)
print(C7.coker_proj(CH_2)==CH_2)

## Mini tests

In [43]:
# str_func test

TF1=eye(len(T.basis))
for i in range(2,len(T.basis)):
    TF1[i-2,i]=1
TK1=str_func(T,TF1,K0)

for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        ad_ij1=SF_ad(T.elt(list(TF1.col(i))),T.elt(list(TF1.col(j))),K0)
        w=zeros(len(T.basis),1)
        for k in range(len(T.basis)):
            key=(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])
            if key in TK1.coeff_dict: 
                w+=Matrix([A*TK1.coeff_dict[key] for A in TF1.col(k)])       
        ad_ij2=sum([w[l]*T.basis[l] for l in range(len(T.basis))])
        if ad_ij1!=ad_ij2:
            print('\n\n\nFailure at', (i,j),':')
            print('ad_ij1 =', ad_ij1,'\n\nad_ij2 =',ad_ij2)
            print('\nDifference:',ad_ij1-ad_ij2)

In [ ]:
# SF_ad test

T_SF=C.cochain()
for i in range(len(T.basis)):
    for j in range(i,len(T.basis)):
        v=T.basis[i].ad(T.basis[j])
        for k in range(len(T.basis)):
            T_SF=T_SF+C.cochain({(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k]):v.vec_rep[k]})

# Test: Does T_SF give the same ad as ad does?

A=4*T.basis[0]-2*T.basis[3]
B=T.basis[1]+3*T.basis[7]+T.basis[3]
A.ad(B)==SF_ad(A,B,T_SF)

## Old Code (to be removed)

In [ ]:
# # Lift to P0 in an invariant way and compute the new structure function
# # V lifts to exp(L(V))*V, where L_H(V)h+L_E(V)e, and V is the lift horizont w.r.t coords (h,e)

# LV=[exp(T.basis[1].ad(T.basis[i]).vec_rep[i]*h+T.basis[2].ad(T.basis[i]).vec_rep[i]*e)
#     for i in range(len(T.basis))]

# # The new structure function

# CH_1=C.cochain()
# for i in range(len(T.basis)):
#     for j in range(len(T.basis)):
#         u=SF_ad(LV[i]*T.basis[i],LV[j]*T.basis[j],T_SF)
#         for k in range(len(T.basis)):
#             k_cmpt=u.vec_rep[k]/LV[k]
#             CH_1+=C.cochain({(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k]):k_cmpt})

In [ ]:
# g,g_coords=C.subspace_proj(CH_1,'im')
# f=C.preim_elt(g_coords)
# CH_1n = CH_1-g # CH_1n is normal


In [ ]:
# # Lift to P1 in an invariant way and compute the new structure function
# # V lifts to exp(L_Y^W(V))W where [Y,V]=L_Y^W(V)W

# # Wildly inefficient, but whatever
# LV=[]
# for i in range(len(T.basis)):
#     LV=LV+[[]]
#     for j in range(len(T.basis)):
#         LV[i]=LV[i]+[[]]
#         u=T.basis[i].ad(T.basis[j])
#         for k in range(len(T.basis)):
#             LV[i][j].append(u.vec_rep[k])
            

In [ ]:
# class ind_add:
#     '''represents a sum of ind_mul objects'''
#     def __init__(self,coeff_dict):
#         # Remove zeros from coeff_dict
#         new_dict=copy.copy(coeff_dict)
#         for k in new_dict:
#             if new_dict[k]==0: del new_dict[k]
#         self.coeff_dict=coeff_dict
        
#     def __eq__(self,other):
#         return self.coeff_dict==other.coeff_dict
        
#     def __add__(self,other):
#         # To do
    
#     def __radd__(self,other):
#         # To do
    
#     def __neg__(self):
#         # To do
        
#     def __sub__(self,other):
#         # To do
    
#     def __mul__(self,k):
#         # To do
    
#     def __rmul__(self,k):
#         # To do
    
#     def __str__(self):
#         # To do
    
#     def __repr__(self):
#         # To do

In [ ]:
# class ind_mul:
#     '''represents a monomial of some ind_der objects'''
#     def __init__(self,exp_dict,coeff=1):
#         if type(exp_dict)==ind_der:
#             self.exp_dict={exp_dict:1}
#             self.coeff=1
#         else:
#             # Remove zeros from exp_dict
#             new_dict=copy.copy(exp_dict)
#             for k in new_dict:
#                 if new_dict[k]==0: del new_dict[k]
#             self.exp_dict=new_dict # represents exponents
#             self.coeff=coeff

#     def __eq__(self,other):
#         if other==0:
#             if self.coeff==0: return True
#         if other==1:
#             for val in self.exp_dict.values():
#                 if val!=0: return False
#         return self.exp_dict==other.exp_dict and self.coeff==other.coeff
    
#     def __mul__(self,k):
#         if type(other)==int or type(other)==float:
#             return ind_mul(self.exp_dict,self.coeff*other)
#         if type(other)==ind_der:
#             if other in self.exp_dict:
#                 new_dict=copy.copy(self.exp_dict)
#                 new_dict[other]=new_dict[other]+1
#             return ind_mul(new_dict,self.coeff)
#         if type(other)==ind_mul:
#             new_dict=copy.copy(self.exp_dict)
#             for k in other.exp_dict:
#                 if k in new_dict: new_dict[k]=new_dict[k]+other.exp_dict[k]
#                 else: new_dict[k]=other.exp_dict[k]
#             return ind_mul(new_dict,self.coeff*other.coeff)
#         if type(other)==ind_add:
#             return other*self
    
#     def __rmul__(self,other):
#         return self*other
        
#     def __add__(self,other):
#         if type(other)==int or type(other)==float:
#             return self+ind_mul({},other)
#         if type(other)==ind_der:
#             return self+ind_mul({other:1},1)
#         if type(other)==ind_mul:
#             return self+ind_add({other:1})
#         if type(other)==ind_add:
#             return other+self
    
#     def __radd__(self,other):
#         return self+other
    
#     def __neg__(self):
#         return ind_mul(self.exp_dict,-self.coeff)
        
#     def __sub__(self,other):
#         return self+(-other)
    
#     def __str__(self):
#         if self.coeff==0:
#             return ''
#         result=''
#         if self.coeff!=1:
#             result+=str(self.coeff)
#         for k in self.exp_dict:
#             result+=' '+str(k)
#             if self.exp_dict[k]!=1:
#                 result+='**'+self.exp_dict[k]
#         return result
    
#     def __repr__(self):
#         if self.coeff==0:
#             return ''
#         result=''
#         if self.coeff!=1:
#             result+=str(self.coeff)
#         for k in self.exp_dict:
#             result+=' '+str(k)
#             if self.exp_dict[k]!=1:
#                 result+='**'+self.exp_dict[k]
#         return result

In [ ]:
# class ind_der:
#     def __init__(self,ind_obj,der_list=[]):
#         self.ind_obj=ind_obj
#         self.der_mset=Multiset(der_list)
        
#     def __eq__(self,other):
#         return (self.ind_obj == other.ind_obj and self.der_set == other.der_set)
        
#     def __add__(self,other):
#         if type(other)==ind_der:
#             return ind_mul(self)+other
#         if type(other)==ind_mul:
#             return other+self
#         if type(other)==ind_add:
#             return other+self
            
#     def __mul__(self,other):
#         if type(other)==int or type(other)==float:
#             return ind_mul({self:1},other)
#         if type(other)==ind_der:
#             return ind_mul(self)*other
#         if type(other)==ind_mul:
#             return other*self
#         if type(other)==ind_add:
#             return other*self
    
#     def __rmul__(self,other):
#         return self*other
            
#     def __radd__(self,other):
#         return self+other
    
#     def __neg__(self):
#         return -ind_mul(self)
        
#     def __sub__(self,other):
#         return other+(-self)
    
#     def __str__(self):
#         der_str=str(self.der_mset)
#         ind_ob_str=str(self.ind_obj)
#         return ind_ob_str[:len(ind_ob_str)-1]+'; '+der_str[1:len(der_str)-1]+ind_ob_str[len(ind_ob_str)-1:]
    
#     def __repr__(self):
#         der_str=str(self.der_mset)
#         ind_ob_str=str(self.ind_obj)
#         return ind_ob_str[:len(ind_ob_str)-1]+'; '+der_str[1:len(der_str)-1]+ind_ob_str[len(ind_ob_str)-1:]
    